In [4]:
import pandas as pd
import random
import datetime

# Helper function to generate random names
def generate_random_name():
    first_names = ['Liam', 'Olivia', 'Noah', 'Emma', 'Oliver', 'Ava', 'Elijah', 'Charlotte', 'William', 'Sophia', 'James', 'Amelia', 'Benjamin', 'Isabella', 'Lucas', 'Mia', 'Henry', 'Evelyn', 'Alexander', 'Harper']
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzales', 'Wilson', 'Anderson', 'Thomas', 'Jackson', 'White', 'Harris', 'Martin']
    return f"{random.choice(first_names)} {random.choice(last_names)}"

# Define the date range for all data
start_date = pd.to_datetime('2025-01-01')
end_date = pd.to_datetime('2025-06-30')

# Number of records
num_orders = 1000
num_customers = 100

# Generate Orders
orders = pd.DataFrame({
    'order_id': [f'O{i:04d}' for i in range(1, num_orders + 1)],
    'customer_id': [f'C{random.randint(1, num_customers):03d}' for _ in range(num_orders)],
    'order_date': [start_date + pd.Timedelta(days=random.randint(0, (end_date - start_date).days)) for _ in range(num_orders)]
})
orders.to_csv('orders.csv', index=False)

# Generate Customers
customers = pd.DataFrame({
    'customer_id': [f'C{i:03d}' for i in range(1, num_customers + 1)],
    'name': [generate_random_name() for _ in range(num_customers)],
    'email': [f'{name.lower().replace(" ", ".")}@example.com' for name in [generate_random_name() for _ in range(num_customers)]] # Ensure unique email prefixes
})
customers.to_csv('customers.csv', index=False)

# Generate Shipments
statuses = ['shipped', 'delivered', 'pending', 'returned']
shipments_data = {
    'shipment_id': [f'S{i:04d}' for i in range(1, num_orders + 1)], # Assuming one shipment per order
    'order_id': orders['order_id'],
    'status': [random.choice(statuses) for _ in range(num_orders)],
    'shipped_at': [start_date + pd.Timedelta(days=random.randint(0, (end_date - start_date).days)) for _ in range(num_orders)]
}

shipments = pd.DataFrame(shipments_data)

# Calculate delivered_at based on shipped_at and status
# Initialize delivered_at with NaT (Not a Time)
shipments['delivered_at'] = pd.NaT

for i in range(num_orders):
    if shipments.loc[i, 'status'] == 'delivered':
        shipped_date = shipments.loc[i, 'shipped_at']
        # Ensure delivery date is after shipment date and within overall date range
        delivery_offset_days = random.randint(1, 10)
        potential_delivered_at = shipped_date + pd.Timedelta(days=delivery_offset_days)
        shipments.loc[i, 'delivered_at'] = min(potential_delivered_at, end_date)
    else:
        shipments.loc[i, 'delivered_at'] = pd.NA # Explicitly set to pd.NA for non-delivered statuses

shipments.to_csv('shipments.csv', index=False)